## Quantamental Long-Short Equity Strategy ##

* This strategy is based on the AQR white paper: "The Case for Core Equity" https://www.anderson.ucla.edu/documents/sites/about/clubs/asam/4-AQR-A-New-Core-Equity-Paradigm.pdf 

* S&P500 constituents, stock prices and financial data are sourced from WRDS, a reliable and academic data source.

* Overview: The strategy aims to rank and score stocks based on Value, Momentum, and Quality (Profitability) factors, constructing a long-only portfolio of only the highest scoring stocks.

* Original testing period: 1980 - 2012.

#### Strategy: ####

1.  **Choose a stock universe** (US 1000 large cap, US 2000 small cap, International large cap equities (Top 85% of market cap from twenty developed markets))

2. **Rank Stocks based on these factors:**

    * **Value.** Value is computed using: 1. P/B ratio; 2. P/E ratio; 3. Forward P/E ratio; 4. EV / FCF; 5. EV / Sales

    * **Quality** (profitability). Profitability is computed using: 1. ROA; 2. FCF-to-Assets; 3. Gross Profit Margins

    * **Momentum.** Momentum is computed using: 1. Trailing 12-1 month return*; 2. 3-day average returns around earnings announcements over prior year

3. **Scoring**
    - Standardize and average metrics within each factor
    - Compute stock-level ranks for Value, Quality, and Momentum

4. **Use a 40%, 40%, 20% weighting on Value, Momentum, and Profitability.**

5. **Select top 25% of ranked stocks and weight them 50/50 based on market cap and score.**

6. **Recalculate the stock ranks and rebalance the portfolio quarterly.**



Note: The 12-1 momentum definition uses a one-month skip to reduce short-term reversal effects (Chan, Jegadeesh, and Lakonishok, 1997).


#### My Strategy Modifications ####
* Use FCF/OCF, ROE, and ROA for the profitability score
* Use P/B ratio and P/E ratio for the value score 
* Use trailing 1 month return only for the momentum score
* Long the top 15%  and short the bottom 15% scores for market neutrality
* Add short book of 10% of the bottom scoring stocks
* Use Universe of 500 stocks (S&P500)
* Rebalance monthly instead of quarterly

### Future Improvements

- Implement global minimum variance optimization on ranked stocks
- Replace trailing P/E with forward P/E
- Add earnings-announcement return effects to the momentum factor
- Test 12-1 month momentum factor & quarterly rebalancing
- Implement a small 10% weighting for sentiment

In [1]:
# Core data wrangling
import pandas as pd
import numpy as np
from pathlib import Path

# WRDS access
import wrds

# Plotting backtest results
import matplotlib.pyplot as plt

# Stats for z-scoring / winsorizing
from scipy import stats

### 1. Data Pulling & Cleaning ###

In [2]:
# Time
START = "2015-01-01"
END   = "2025-12-31"

# Factor weights (must sum to 1)
W_VALUE    = 0.40
W_MOMENTUM = 0.40
W_QUALITY  = 0.20

# Portfolio sizing
LONG_PCT  = 0.20   # top 20% of scores go long
SHORT_PCT = 0.20   # bottom 20% go short

# Rebalance: month-end
REBAL = "M"

# Cache folder for WRDS pulls
CACHE = Path("./wrds_cache"); CACHE.mkdir(exist_ok=True)

# Connect to WRDS (prompts for username/password the first time only)
db = wrds.Connection()

# Tiny cache helper: pull once, then read parquet on reruns
def pull(name, sql, date_cols=None):
    f = CACHE / f"{name}.parquet"
    if f.exists():
        return pd.read_parquet(f)
    df = db.raw_sql(sql, date_cols=date_cols)
    df.to_parquet(f)
    return df

WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [3]:
(CACHE / "ratios.parquet").unlink(missing_ok=True)

In [4]:
# --- 1. S&P 500 historical membership ---
sp500 = pull("sp500", f"""
    SELECT permno, start, ending
    FROM crsp.dsp500list
    WHERE ending >= '{START}'
""", date_cols=["start", "ending"])

# --- 2. Monthly prices (with ticker + common-stock filter) ---
permnos = tuple(int(p) for p in sp500.permno.unique())
prices = pull("prices", f"""
    SELECT d.permno, n.ticker, d.date, d.prc, d.ret, d.shrout
    FROM crsp.msf AS d
    INNER JOIN crsp.msenames AS n
      ON d.permno = n.permno
     AND d.date >= n.namedt
     AND d.date <= n.nameendt
    WHERE d.permno IN {permnos}
      AND d.date BETWEEN '{START}' AND '{END}'
      AND n.shrcd IN (10, 11)
      AND n.exchcd IN (1, 2, 3)
""", date_cols=["date"])

# --- 3. Pre-computed financial ratios from WRDS Financial Ratios Suite ---
ratios = pull("ratios", f"""
    SELECT permno, public_date,
           ptb, pe_exi,
           roe, roa, fcf_ocf
    FROM wrdsapps.firm_ratio
    WHERE permno IN {permnos}
      AND public_date BETWEEN '{START}' AND '{END}'
""", date_cols=["public_date"])

# --- Sanity checks ---
print(f"sp500  rows: {len(sp500):>8,}   unique permnos: {sp500.permno.nunique():,}")
print(f"prices rows: {len(prices):>8,}   range: {prices.date.min().date()} -> {prices.date.max().date()}")
print(f"ratios rows: {len(ratios):>8,}   unique permnos: {ratios.permno.nunique():,}")
print("\nratios sample:"); print(ratios.head(3))


sp500  rows:      741   unique permnos: 736
prices rows:   66,513   range: 2015-01-30 -> 2024-12-31
ratios rows:   71,656   unique permnos: 642

ratios sample:
   permno public_date       ptb     pe_exi       roe       roa  fcf_ocf
0   10104  2015-01-31  3.869221  17.454167  0.238477  0.199364   0.9524
1   10104  2015-02-28  4.047488  18.258333  0.238477  0.199364   0.9524
2   10104  2015-03-31   3.96355  17.979167  0.238477  0.199364   0.9524


In [27]:
# --- Membership spine: (permno, month_end) for every month in the index ---
month_ends = pd.date_range(START, END, freq='ME')

spine_rows = []
for _, r in sp500.iterrows():
    valid = month_ends[(month_ends >= r['start']) & (month_ends <= r['ending'])]
    spine_rows.extend((r['permno'], d) for d in valid)
spine = pd.DataFrame(spine_rows, columns=['permno', 'month_end'])

spine['ym']  = spine['month_end'].dt.to_period('M')
prices['ym'] = prices['date'].dt.to_period('M')
ratios['ym'] = ratios['public_date'].dt.to_period('M')

panel = (
    spine
    .merge(prices[['permno', 'ticker', 'ym', 'prc', 'ret', 'shrout']],
           on=['permno', 'ym'], how='left')
    .merge(ratios[['permno', 'ym', 'ptb', 'pe_exi', 'roe', 'roa', 'fcf_ocf']],
           on=['permno', 'ym'], how='left')
    .drop(columns='ym')
    .sort_values(['permno', 'month_end'])
    .reset_index(drop=True)
)

panel['mktcap'] = panel['prc'].abs() * panel['shrout']

# Sanity checks
print(f"panel shape: {panel.shape}")
print(f"unique permnos: {panel.permno.nunique()}")
print(f"date range: {panel.month_end.min().date()} -> {panel.month_end.max().date()}")
print(f"% with price:    {panel['prc'].notna().mean():.1%}")
print(f"% with ptb:      {panel['ptb'].notna().mean():.1%}")
print(f"% with roe:      {panel['roe'].notna().mean():.1%}")
print(f"% with fcf_ocf:  {panel['fcf_ocf'].notna().mean():.1%}")


panel shape: (60579, 12)
unique permnos: 712
date range: 2015-01-31 -> 2024-12-31
% with price:    88.5%
% with ptb:      82.3%
% with roe:      82.4%
% with fcf_ocf:  85.0%


In [28]:
# print("=== shape & dtypes ===")
# panel.info()

print("\n=== head ===")
print(panel.head(10))

# print("\n=== Apple (permno 14593) ===")
# print(panel[panel.permno == 14593].head(15))

print("\n=== coverage by year ===")
print(panel.groupby(panel.month_end.dt.year).agg(
    rows=('permno', 'size'),
    stocks=('permno', 'nunique'),
    pct_priced=('prc', lambda x: x.notna().mean()),
    pct_ptb=('ptb', lambda x: x.notna().mean()),
    pct_roe=('roe', lambda x: x.notna().mean()),
    pct_fcf_ocf=('fcf_ocf', lambda x: x.notna().mean()),
).round(3))

# print("\n=== summary stats ===")
# print(panel[['prc', 'ret', 'mktcap', 'ptb', 'pe_exi', 'roe', 'roa', 'fcf_ocf']].describe().round(3))


=== head ===
   permno  month_end ticker    prc       ret     shrout       ptb     pe_exi  \
0   10104 2015-01-31   ORCL  41.89 -0.065822  4391367.0  3.869221  17.454167   
1   10104 2015-02-28   ORCL  43.82  0.046073  4391367.0  4.047488  18.258333   
2   10104 2015-03-31   ORCL  43.15  -0.01529  4367070.0   3.96355  17.979167   
3   10104 2015-04-30   ORCL  43.62  0.014368  4698310.0  4.261156  18.251046   
4   10104 2015-05-31   ORCL  43.49  -0.00298  4367070.0  3.948932  18.196653   
5   10104 2015-06-30   ORCL   40.3  -0.07335  4336077.0  3.633307  16.861925   
6   10104 2015-07-31   ORCL  39.94 -0.005211  4336077.0  3.531246  18.072398   
7   10104 2015-08-31   ORCL  37.09 -0.071357  4336077.0  3.279267  16.782805   
8   10104 2015-09-30   ORCL  36.12 -0.026153  4264628.0  3.140884  16.343891   
9   10104 2015-10-31   ORCL  38.84  0.079457  4264628.0  3.516958  18.149533   

        roe       roa   fcf_ocf        mktcap  
0  0.238477  0.199364    0.9524  183954363.63  
1  0.2384

In [29]:
panel['month_end'] = pd.to_datetime(panel['month_end'])
panel.set_index(['month_end', 'permno'], inplace=True)
panel.sort_index(inplace=True)
panel

ticker     prc       ret     shrout        ptb      pe_exi  \
month_end  permno                                                              
2015-01-31 10104    ORCL   41.89 -0.065822  4391367.0   3.869221   17.454167   
           10107    MSFT    40.4 -0.130248  8203785.0   3.568245   15.843137   
           10138    TROW   78.72 -0.083159   262073.0   3.946511   17.769752   
           10145     HON   97.76 -0.021617   782810.0   3.859568    18.37594   
           10147     EMC   25.93  -0.12811  2034909.0   2.290951    20.91129   
...                  ...     ...       ...        ...        ...         ...   
2024-12-31 93096      DG   75.82 -0.018765   219926.0   1.966211   12.490939   
           93132    FTNT   94.48 -0.005997   766453.0  61.571846   47.717172   
           93246    GNRC  155.05 -0.176142    59497.0   3.732255    32.16805   
           93429    <NA>    <NA>      <NA>       <NA>   4.643522   26.621253   
           93436    TSLA  403.84  0.170008  3210060.0  18.574856  109.145946   

                        roe       roa   fcf_ocf        mktcap  
month_end  permno                                              
2015-01-31 10104   0.238477  0.199364    0.9524  183954363.63  
           10107   0.241737  0.216786   0.82903   331432914.0  
           10138   0.244281  0.368142  0.911851   20630386.56  
           10145   0.234596  0.166222  0.780933    76527505.6  
           10147   0.113554  0.136272  0.851435   52765190.37  
...                     ...       ...       ...           ...  
2024-12-31 93096   0.166753  0.094672  0.524057   16674789.32  
           93132   8.445487  0.226958  0.843728   72414479.44  
           93246   0.120684  0.129868  0.812545    9225009.85  
           93429   0.187383  0.159717  0.966584          <NA>  
           93436   0.208738  0.126242  0.249327  1296350630.4  

[60579 rows x 10 columns]

### 2. Trading Algorithm ###

In [ ]:
def rank_value(df):
    

def rank_momentum(df):

def rank_profitability(df):

IndentationError: expected an indented block after function definition on line 1 (3847084942.py, line 4)

### 3. Backtesting ###

* Use python bt trade metric library?